<a href="https://colab.research.google.com/github/cityhunter0831/sar-atr/blob/claude%2Fwork-progress-summary-afnetj/notebooks/colab_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAR-ATR Colab 실행 템플릿

**실행 순서**:
Cell 1(환경설정) → Cell 2(동작확인) → Cell 3(MSTAR 압축해제) → Cell 4(gengzhe zip) → Cell 5(SAMPLE)
→ Cell 6(Exp A) → Cell 6vis(Exp A 그래프) → Cell 6b(SSIM)
→ Cell 7a(Exp B MATLAB ⭐) → Cell 7b(Grad-CAM) → Cell 7c(XAI) → Cell 7vis(XAI 그래프)
→ Cell 8(Exp C) → Cell 8vis(Exp C 그래프)
→ Cell 9(Exp D) → Cell 9vis(Exp D ROC)
→ Cell 10(Drive 저장 + 커밋)

**GPU 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택

> **주의**: Cell 1의 `TOKEN`을 본인의 GitHub Personal Access Token으로 교체하세요.

> **gengzhe 데이터**: Drive에 zip 파일이 있어야 합니다. Cell 4에서 자동 압축 해제.

> **Exp B 핵심**: Cell 7a가 논문 재현(90.9%). MATLAB .mat 파일이 `MyDrive/SAR_ATR_Project/data/mstar_data_aug/`에 있어야 함.

In [ ]:
# ── Cell 1: 환경 설정 ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SAR_ATR_Project'
TOKEN = 'YOUR_GITHUB_TOKEN_HERE'  # Personal Access Token (repo scope)
BRANCH = 'claude/work-progress-summary-afnetj'

import os, sys

if not os.path.exists('/content/repo'):
    !git clone -b {BRANCH} https://{TOKEN}@github.com/cityhunter0831/sar-atr.git /content/repo
else:
    %cd /content/repo
    !git pull origin {BRANCH}

%cd /content/repo
!pip install -r requirements.txt -q
!pip install optuna -q
sys.path.insert(0, '/content/repo')

os.makedirs(f'{DRIVE_ROOT}/data',    exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/figures', exist_ok=True)

if not os.path.exists('/content/data'):
    !cp -r {DRIVE_ROOT}/data /content/data
    print('데이터 복사 완료')
else:
    print('데이터 이미 존재 — 스킵')

if not os.path.exists('/content/repo/data'):
    os.symlink('/content/data', '/content/repo/data')
if not os.path.exists('/content/repo/results'):
    os.symlink(f'{DRIVE_ROOT}/results', '/content/repo/results')

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
# ── Cell 2: 파이프라인 동작 확인 (mock 데이터) ───────────────────────
from core.mock_data import MockSARDataset
from core.models import get_model
from core.train import train_model
from core.interfaces import TrainConfig

train_ds = MockSARDataset(n=200, num_classes=3, seed=0)
val_ds   = MockSARDataset(n=60,  num_classes=3, seed=1)

for model_name in ['smpl', 'resnet18']:
    model  = get_model(model_name, num_classes=3)
    config = TrainConfig(model_name=model_name, num_classes=3, epochs=5, seed=0)
    model, result = train_model(model, train_ds, val_ds, config)
    print(f'[{model_name}] mock val acc = {result.accuracy*100:.1f}%')

print('\n✅ 파이프라인 정상')

In [ ]:
# ── Cell 3: MSTAR ZIP 압축 풀기 ──────────────────────────────────────
import zipfile, os

MSTAR_DIR   = '/content/data/mstar'
DRIVE_MSTAR = f'{DRIVE_ROOT}/data/mstar'
os.makedirs(MSTAR_DIR, exist_ok=True)

zip_names = [
    'MSTAR-PublicTargetChips-T72-BMP2-BTR70-SLICY.zip',
    'MSTAR-PublicMixedTargets-CD1.zip',
    'MSTAR-PublicMixedTargets-CD2.zip',
]

for fname in zip_names:
    src = f'{DRIVE_MSTAR}/{fname}'
    if not os.path.exists(src):
        print(f'Drive에 없음 (스킵): {fname}')
        continue
    print(f'압축 해제 중: {fname}')
    with zipfile.ZipFile(src) as z:
        z.extractall(MSTAR_DIR)
    print('  → 완료')

# SAR-ship
SARSHIP_DIR = '/content/data/sarship'
os.makedirs(SARSHIP_DIR, exist_ok=True)
sarship_zip = f'{DRIVE_ROOT}/data/sarship/ship_dataset_v0.zip'
if os.path.exists(sarship_zip):
    with zipfile.ZipFile(sarship_zip) as z:
        z.extractall(SARSHIP_DIR)
    import glob
    imgs = glob.glob(f'{SARSHIP_DIR}/**/*.png', recursive=True)
    print(f'SAR-ship: {len(imgs)}장')
else:
    print('SAR-ship ZIP 없음 (스킵)')

In [ ]:
# ── Cell 4: gengzhe2015 데이터 ZIP 압축 해제 (Exp A용) ───────────────
# Drive에 zip 파일 4개가 있어야 함:
#   Original MSTAR Images.zip / Train_OR_Test_CT.zip
#   Train_CT_Test_CT.zip / Train_CTx2_Test_CT.zip
import zipfile, os
from pathlib import Path

DRIVE_GENG = f'{DRIVE_ROOT}/data/clutter_gengzhe'
LOCAL_GENG = '/content/data/clutter_gengzhe'
os.makedirs(LOCAL_GENG, exist_ok=True)

geng_zips = [
    'Original MSTAR Images.zip',
    'Train_OR_Test_CT.zip',
    'Train_CT_Test_CT.zip',
    'Train_CTx2_Test_CT.zip',
]

for fname in geng_zips:
    zpath = f'{DRIVE_GENG}/{fname}'
    if not os.path.exists(zpath):
        print(f'없음 (스킵): {fname}')
        continue
    print(f'압축 해제 중: {fname}')
    with zipfile.ZipFile(zpath, 'r') as z:
        z.extractall(LOCAL_GENG)
    print(f'  → 완료')

# 클래스별 장수 확인
print('\n── 폴더별 클래스 장수 ─────────────────────────────')
for folder in sorted(Path(LOCAL_GENG).iterdir()):
    if not folder.is_dir(): continue
    cls_counts = {}
    for cls in sorted(folder.iterdir()):
        if cls.is_dir():
            cls_counts[cls.name] = len(list(cls.glob('*')))
    if cls_counts:
        total = sum(cls_counts.values())
        print(f'  {folder.name}: {total}장 {cls_counts}')
print('────────────────────────────────────────────────────')

In [ ]:
# ── Cell 5: SAMPLE 데이터셋 받기 (Exp C/D용) ────────────────────────
import os
SAMPLE_DST = f'{DRIVE_ROOT}/data/sample'

if not os.path.exists(f'{SAMPLE_DST}/png_images'):
    !git clone https://github.com/benjaminlewis-afrl/SAMPLE_dataset_public /content/sample_repo
    os.makedirs(SAMPLE_DST, exist_ok=True)
    !cp -r /content/sample_repo/png_images {SAMPLE_DST}/png_images
    print('SAMPLE 데이터 복사 완료')
else:
    print('이미 존재함 — 스킵')

if not os.path.exists('/content/data/sample'):
    !cp -r {SAMPLE_DST} /content/data/sample

!ls /content/data/sample/png_images | head -5

In [ ]:
# ── Cell 6: Exp A — 클러터 전이 학습 (Table 4) ───────────────────────
import sys, importlib
for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments','gradcam'): del sys.modules[m]
importlib.invalidate_caches()

from experiments.exp_a_clutter_transfer import run_all
results_a = run_all(epochs=60)
print(results_a)

In [ ]:
# ── Cell 6vis: Exp A 시각화 — 막대 그래프 ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

FIG_DIR = Path('/content/repo/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

conditions = ['MSTAROR', 'TrainOR\n+TestCT', 'TrainCT\n+TestCT', 'TrainCTx2\n+TestCT']
cond_keys  = ['MSTAROR', 'TrainOR+TestCT', 'TrainCT+TestCT', 'TrainCTx2+TestCT']

paper_smpl = [98.1, 38.6, 91.5, 96.0]
paper_rn18 = [99.8, 55.2, 97.5, 98.4]

our_smpl = [results_a['smpl'][k]['mean'] for k in cond_keys]
our_rn18 = [results_a['resnet18'][k]['mean'] for k in cond_keys]
our_smpl_std = [results_a['smpl'][k]['std'] for k in cond_keys]
our_rn18_std = [results_a['resnet18'][k]['std'] for k in cond_keys]

x = np.arange(len(conditions))
w = 0.2

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - 1.5*w, paper_smpl, w, label='논문 SMPL7',   color='#94b8e0', edgecolor='white')
b2 = ax.bar(x - 0.5*w, our_smpl,   w, label='우리 SMPL',    color='#2176ae', edgecolor='white',
            yerr=our_smpl_std, capsize=3, error_kw={'elinewidth':1.2})
b3 = ax.bar(x + 0.5*w, paper_rn18, w, label='논문 RN18',    color='#f4a261', edgecolor='white')
b4 = ax.bar(x + 1.5*w, our_rn18,   w, label='우리 RN18',    color='#e76f51', edgecolor='white',
            yerr=our_rn18_std, capsize=3, error_kw={'elinewidth':1.2})

# 도메인 갭 → 회복 화살표
ax.annotate('', xy=(1-0.5*w, our_smpl[1]+3), xytext=(0-0.5*w, our_smpl[0]-3),
            arrowprops=dict(arrowstyle='->', color='#2176ae', lw=1.5))
ax.text(0.5, 72, '도메인 갭', ha='center', fontsize=8, color='#2176ae')
ax.annotate('', xy=(2-0.5*w, our_smpl[2]-3), xytext=(1-0.5*w, our_smpl[1]+3),
            arrowprops=dict(arrowstyle='->', color='#2176ae', lw=1.5))
ax.text(1.5, 85, '회복', ha='center', fontsize=8, color='#2176ae')

ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=10)
ax.set_ylabel('정확도 (%)', fontsize=11)
ax.set_title('Exp A — 클러터 전이 재현 (Table 4)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 108)
ax.legend(fontsize=9, loc='lower right')
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# 값 레이블
for bar_group in [b2, b4]:
    for bar in bar_group:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+1.5, f'{h:.1f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'exp_a_clutter_transfer.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {FIG_DIR}/exp_a_clutter_transfer.png')

In [ ]:
# ── Cell 6b: Exp A — SSIM 경계 아티팩트 정량화 (개선 #1) ─────────────
from experiments.exp_a_clutter_transfer import run_boundary_ssim_analysis
ssim_result = run_boundary_ssim_analysis(n_samples=20)
print(ssim_result)

In [ ]:
# ── Cell 7a: Exp B — MATLAB 희소복원 증강 (논문 재현 90.9%) ⭐ ─────────
import os, sys, importlib, torch
from pathlib import Path

for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments','gradcam'): del sys.modules[m]
importlib.invalidate_caches()

AUG_DATA_DIR = '/content/data/mstar_data_aug'
if not os.path.exists(AUG_DATA_DIR):
    drive_aug = f'{DRIVE_ROOT}/data/mstar_data_aug'
    if os.path.exists(drive_aug):
        !cp -r {drive_aug} {AUG_DATA_DIR}
        print(f'Drive → {AUG_DATA_DIR} 복사 완료')
    else:
        print('⚠️ MATLAB 증강 데이터 없음 — Cell 7a 스킵')
        AUG_DATA_DIR = None

if AUG_DATA_DIR:
    from augmentation.precomputed_aug import AugImagesDataset, BaselineDataset, TestImagesDataset
    from core.models import get_model
    from core.train import train_model
    from core.interfaces import TrainConfig
    from core.evaluate import evaluate

    test_ds     = TestImagesDataset(AUG_DATA_DIR, log_scale=True, dyn_range_db=60)
    baseline_ds = BaselineDataset(AUG_DATA_DIR,   log_scale=True, dyn_range_db=60)
    aug_ds      = AugImagesDataset(AUG_DATA_DIR,  log_scale=True, dyn_range_db=60)
    print(f'Baseline: {len(baseline_ds)}장 / Aug: {len(aug_ds)}장 / Test: {len(test_ds)}장')

    cfg = TrainConfig(model_name='smpl', num_classes=5, epochs=60, loss_type='at')

    model_bl = get_model('smpl', 5)
    _, result_bl = train_model(model_bl, baseline_ds, test_ds, cfg)
    eval_bl = evaluate(model_bl, test_ds)
    print(f'\nBaseline: {result_bl.accuracy*100:.1f}%  (논문 56.6%)')
    print(f'Per-class: {eval_bl.per_class_accuracy}')

    model_aug = get_model('smpl', 5)
    _, result_aug = train_model(model_aug, aug_ds, test_ds, cfg)
    eval_aug = evaluate(model_aug, test_ds)
    print(f'\nAug (scattering): {result_aug.accuracy*100:.1f}%  (논문 96.4%)')
    print(f'Per-class: {eval_aug.per_class_accuracy}')

    # 체크포인트 저장 (Grad-CAM용)
    ckpt_dir = Path('/content/repo/results/exp_b')
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model_aug.state_dict(), ckpt_dir / 'smpl_ph_aug.pth')

    # confusion matrix 저장용
    import numpy as np
    np.save(ckpt_dir / 'cm_baseline.npy', eval_bl.confusion_matrix)
    np.save(ckpt_dir / 'cm_aug.npy',      eval_aug.confusion_matrix)
    CLASS_NAMES = test_ds.class_names
    print(f'\n체크포인트 + confusion matrix 저장: {ckpt_dir}')

In [ ]:
# ── Cell 7a-vis: Exp B — Confusion Matrix (baseline vs aug) ──────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path('/content/repo/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
ckpt_dir = Path('/content/repo/results/exp_b')

cm_bl  = np.load(ckpt_dir / 'cm_baseline.npy')
cm_aug = np.load(ckpt_dir / 'cm_aug.npy')

# CLASS_NAMES는 Cell 7a에서 정의됨. 없으면 기본값 사용
try:
    labels = CLASS_NAMES
except NameError:
    labels = ['2S1','BMP2','BTR70','T72','ZSU23']

def plot_cm(ax, cm, title, acc):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.set_title(f'{title}\n(Acc {acc:.1f}%)', fontsize=11, fontweight='bold')
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = cm_norm[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)
    return im

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
acc_bl  = np.diag(cm_bl).sum()  / cm_bl.sum()  * 100
acc_aug = np.diag(cm_aug).sum() / cm_aug.sum() * 100
plot_cm(axes[0], cm_bl,  'Baseline (136장, 증강 없음)', acc_bl)
im = plot_cm(axes[1], cm_aug, 'Aug (MATLAB 희소복원 증강)', acc_aug)
fig.colorbar(im, ax=axes, shrink=0.8, label='Recall')
fig.suptitle('Exp B — Confusion Matrix (SMPL/AT, El17→15°)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp_b_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {FIG_DIR}/exp_b_confusion_matrix.png')

In [ ]:
# ── Cell 7b: Grad-CAM (8×8 + 16×16) ─────────────────────────────────
import sys, importlib
!cd /content/repo && git pull
for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments','gradcam'): del sys.modules[m]
importlib.invalidate_caches()

from experiments.exp_b_ph_scattering import run_gradcam_analysis
from pathlib import Path

print('=== Grad-CAM 8×8 (마지막 conv) ===')
gc_8 = run_gradcam_analysis(model_name='smpl', n_samples=10,
                             log_scale=True, dyn_range_db=60,
                             cam_from_last=0,
                             save_dir=Path('results/exp_b/gradcam_8x8'))

print('\n=== Grad-CAM 16×16 (from_last=1) ===')
gc_16 = run_gradcam_analysis(model_name='smpl', n_samples=10,
                              log_scale=True, dyn_range_db=60,
                              cam_from_last=1,
                              save_dir=Path('results/exp_b/gradcam_16x16'))

import glob
from IPython.display import Image as IPImage, display
for png in sorted(glob.glob('results/exp_b/gradcam_16x16/*.png'))[:3]:
    print(png); display(IPImage(png))

In [ ]:
# ── Cell 7c: Occlusion + SmoothGrad-IG (픽셀 단위 XAI) ───────────────
import sys, importlib
for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments','gradcam'): del sys.modules[m]
importlib.invalidate_caches()

from experiments.exp_b_ph_scattering import run_xai_analysis
xai_records = run_xai_analysis(
    model_name='smpl', n_samples=10,
    log_scale=True, dyn_range_db=60,
    methods=('occlusion','smoothgrad_ig'),
)

import glob
from IPython.display import Image as IPImage, display
for png in sorted(glob.glob('results/exp_b/xai/*_xai.png'))[:3]:
    print(png); display(IPImage(png))

In [ ]:
# ── Cell 7vis: XAI IoU 단조 증가 그래프 ──────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

FIG_DIR = Path('/content/repo/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 실험 결과 수치 (REPORT.md 확정값)
methods  = ['Grad-CAM\n8×8', 'Grad-CAM\n16×16', 'Occlusion\nSensitivity', 'SmoothGrad-IG']
iou_vals = [0.12, 0.19, 0.24, 0.29]
colors   = ['#94b8e0', '#2176ae', '#f4a261', '#e76f51']
labels_desc = ['CAM 계열\n(8×8 해상도)', 'CAM 계열\n(16×16 해상도)', '인과적\n(픽셀 공간)', '공리적\n(픽셀 공간)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: IoU 막대 그래프
bars = ax1.bar(methods, iou_vals, color=colors, edgecolor='white', width=0.6)
ax1.plot(range(len(methods)), iou_vals, 'k--o', markersize=7, linewidth=1.5, label='단조 증가')
ax1.axhline(0.05, color='gray', linestyle=':', linewidth=1, label='랜덤 기준선 (≈0.05)')
for bar, val in zip(bars, iou_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.2f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, 0.38)
ax1.set_ylabel('산란점 IoU (↑)', fontsize=11)
ax1.set_title('XAI 방법별 산란점 정합도 (IoU)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.yaxis.grid(True, alpha=0.3); ax1.set_axisbelow(True)

# 오른쪽: 방법 특성 비교 표
ax2.axis('off')
table_data = [
    ['방법', '해상도', '성격', 'IoU'],
    ['Grad-CAM 8×8',      '8×8',    'CAM 계열',   '0.12'],
    ['Grad-CAM 16×16',    '16×16',  'CAM 계열',   '0.19'],
    ['Occlusion',         '64×64',  '인과적',     '0.24'],
    ['SmoothGrad-IG',     '64×64',  '공리적',     '0.29 ★'],
]
tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
tbl.scale(1.4, 2.2)
# 헤더 색상
for j in range(4):
    tbl[0, j].set_facecolor('#2176ae'); tbl[0, j].set_text_props(color='white', fontweight='bold')
# 마지막 행 강조
for j in range(4):
    tbl[4, j].set_facecolor('#fff3cd')
ax2.set_title('방법별 특성 비교', fontsize=12, fontweight='bold', pad=15)

fig.suptitle('Exp B 개선 #3 — XAI × 산란점 IoU 검증 (SMPL/AT 90.9%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp_b_xai_iou.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {FIG_DIR}/exp_b_xai_iou.png')

In [ ]:
# ── Cell 8: Exp C — 대비 증강 재현 + Optuna 자동탐색 (개선 #2) ────────
import sys, importlib
!cd /content/repo && git pull
for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments'): del sys.modules[m]
importlib.invalidate_caches()

from experiments.exp_c_contrast_optuna import run as run_c
results_c = run_c(model_name='resnet18', n_optuna_trials=20, epochs_full=60, epochs_trial=10)

In [ ]:
# ── Cell 8vis: Exp C 시각화 — 3단계 막대 + Optuna 탐색 산점도 ─────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

FIG_DIR = Path('/content/repo/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# results_c 딕셔너리에서 수치 추출
acc_noaug  = results_c.get('no_aug_acc',   69.6)
acc_paper  = results_c.get('paper_acc',    61.8)
acc_optuna = results_c.get('optuna_acc',   80.3)
best_params = results_c.get('best_params', {'strength': 0.672, 'levels': 4})
optuna_trials = results_c.get('trials', [])   # list of {'strength':..,'levels':..,'val_acc':..}

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6])

# ── 왼쪽: 3단계 막대 ──
ax1 = fig.add_subplot(gs[0])
steps  = ['① no-aug', '② 논문\nColorJitter(0.5)×3', '③ Optuna\n자동탐색']
accs   = [acc_noaug, acc_paper, acc_optuna]
colors = ['#94b8e0', '#e76f51', '#2a9d8f']
bars = ax1.bar(steps, accs, color=colors, edgecolor='white', width=0.55)
ax1.axhline(91.9, color='#888', linestyle='--', linewidth=1.2, label='논문 Ori 91.9%')
ax1.axhline(94.5, color='#444', linestyle='--', linewidth=1.2, label='논문 Aug 94.5%')

# 차이 화살표
ax1.annotate('', xy=(1, acc_paper+1), xytext=(0, acc_noaug-1),
             arrowprops=dict(arrowstyle='->', color='#e76f51', lw=1.5))
ax1.text(0.5, 67, f'−{acc_noaug-acc_paper:.1f}%p\n(악화)', ha='center', fontsize=8.5, color='#e76f51')
ax1.annotate('', xy=(2, acc_optuna-1), xytext=(1, acc_paper+1),
             arrowprops=dict(arrowstyle='->', color='#2a9d8f', lw=1.5))
ax1.text(1.5, 73, f'+{acc_optuna-acc_paper:.1f}%p\n(회복)', ha='center', fontsize=8.5, color='#2a9d8f')

for bar, val in zip(bars, accs):
    ax1.text(bar.get_x()+bar.get_width()/2, val+0.8, f'{val:.1f}%',
             ha='center', fontsize=10, fontweight='bold')
ax1.set_ylim(50, 100)
ax1.set_ylabel('정확도 (%) — real_test', fontsize=10)
ax1.set_title('Exp C 개선 #2 — 3단계 비교 (RN18, K=0)', fontsize=11, fontweight='bold')
ax1.legend(fontsize=8.5, loc='upper left')
ax1.yaxis.grid(True, alpha=0.3); ax1.set_axisbelow(True)

# ── 오른쪽: Optuna 탐색 산점도 ──
ax2 = fig.add_subplot(gs[1])
if optuna_trials:
    strengths = [t.get('strength', t.get('params', {}).get('strength', 0)) for t in optuna_trials]
    lvls      = [t.get('levels',   t.get('params', {}).get('levels', 1))   for t in optuna_trials]
    val_accs  = [t.get('val_acc',  t.get('value', 0)) * (100 if t.get('val_acc',1) < 2 else 1) for t in optuna_trials]
    sc = ax2.scatter(strengths, val_accs, c=lvls, cmap='RdYlGn', s=80, alpha=0.8,
                     vmin=1, vmax=4, edgecolors='white', linewidths=0.5)
    plt.colorbar(sc, ax=ax2, label='levels', ticks=[1,2,3,4])
    # 최적점 강조
    best_s = best_params.get('strength', 0.672)
    best_l = best_params.get('levels', 4)
    best_v = max(val_accs)
    ax2.scatter([best_s], [best_v], s=200, c='red', marker='*', zorder=5, label=f'최적 (s={best_s:.2f}, l={best_l})')
    ax2.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='논문 strength=0.5')
    ax2.legend(fontsize=8.5)
else:
    # 수치 없으면 안내 텍스트
    ax2.text(0.5, 0.5, 'Optuna trial 데이터를\nresults_c["trials"]에서 불러올 수 없음',
             ha='center', va='center', transform=ax2.transAxes, fontsize=10)
ax2.set_xlabel('strength (대비 흔들기 강도)', fontsize=10)
ax2.set_ylabel('val 정확도 (%)', fontsize=10)
ax2.set_title('Optuna 탐색 공간 (strength × levels)', fontsize=11, fontweight='bold')
ax2.yaxis.grid(True, alpha=0.3); ax2.set_axisbelow(True)

plt.suptitle('Exp C 개선 #2 — 대비 자동 최적화 (SAMPLE synth→real)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp_c_contrast_optuna.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {FIG_DIR}/exp_c_contrast_optuna.png')

In [ ]:
# ── Cell 9: Exp D — OOD 탐지 ─────────────────────────────────────────
import sys, importlib
for m in list(sys.modules):
    if m.split('.')[0] in ('augmentation','core','experiments','gradcam'): del sys.modules[m]
importlib.invalidate_caches()

from experiments.exp_d_ood import run as run_d
results_d = run_d(model_name='smpl', j_list=[1, 2, 3], epochs=60)

In [ ]:
# ── Cell 9vis: Exp D 시각화 — AUROC 히트맵 + 방법 비교 ───────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from pathlib import Path

FIG_DIR = Path('/content/repo/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 확정 수치 (results_d에서 추출 시도, 없으면 기록값 사용)
data = {
    ('odin',        'holdout'): [0.516, 0.473, 0.515],
    ('odin',        'sarship'): [1.000, 0.000, 0.995],
    ('mahalanobis', 'holdout'): [0.386, 0.400, 0.451],
    ('mahalanobis', 'sarship'): [1.000, 1.000, 1.000],
}
# results_d는 list[dict] 구조: results_d[j_idx] = {'odin_holdout_auroc': ..., ...}
if results_d:
    for j_idx, rec in enumerate(results_d):
        for method in ('odin', 'mahalanobis'):
            for ood in ('holdout', 'sarship'):
                val = rec.get(f'{method}_{ood}_auroc')
                if val is not None and j_idx < 3:
                    data[(method, ood)][j_idx] = val

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 왼쪽: AUROC 히트맵 ──
ax = axes[0]
row_labels = ['ODIN\nholdout', 'ODIN\nsarship', 'Maha\nholdout', 'Maha\nsarship']
col_labels = ['J=1\n(m548)', 'J=2\n(m35,m548)', 'J=3\n(m35,m548,t72)']
keys = [('odin','holdout'),('odin','sarship'),('mahalanobis','holdout'),('mahalanobis','sarship')]
matrix = np.array([data[k] for k in keys])

im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='AUROC')
ax.set_xticks(range(3)); ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(4)); ax.set_yticklabels(row_labels, fontsize=10)
ax.set_title('AUROC 히트맵 (녹색=탐지 잘됨)', fontsize=11, fontweight='bold')
for i in range(4):
    for j in range(3):
        val = matrix[i, j]
        note = '†' if (i==1 and j==1) else ''  # J=2 ODIN sarship 이상치
        color = 'white' if val < 0.3 or val > 0.8 else 'black'
        ax.text(j, i, f'{val:.3f}{note}', ha='center', va='center', fontsize=10, color=color, fontweight='bold')

# ── 오른쪽: Far-OOD vs Near-OOD 대조 막대 ──
ax2 = axes[1]
j_vals = [1, 2, 3]
odin_near  = data[('odin',        'holdout')]
maha_near  = data[('mahalanobis', 'holdout')]
odin_far   = data[('odin',        'sarship')]
maha_far   = data[('mahalanobis', 'sarship')]

x = np.arange(3)
w = 0.2
ax2.bar(x - 1.5*w, odin_near, w, label='ODIN / near-OOD',  color='#94b8e0')
ax2.bar(x - 0.5*w, maha_near, w, label='Maha / near-OOD',  color='#2176ae')
ax2.bar(x + 0.5*w, odin_far,  w, label='ODIN / far-OOD',   color='#f4a261')
ax2.bar(x + 1.5*w, maha_far,  w, label='Maha / far-OOD',   color='#e76f51')
ax2.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='랜덤 기준선')
ax2.set_xticks(x); ax2.set_xticklabels([f'J={j}' for j in j_vals], fontsize=10)
ax2.set_ylabel('AUROC', fontsize=11); ax2.set_ylim(0, 1.1)
ax2.set_title('Near-OOD vs Far-OOD 탐지 비교', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8.5, loc='upper right')
ax2.yaxis.grid(True, alpha=0.3); ax2.set_axisbelow(True)
ax2.text(2.5, 0.52, 'near-OOD\n한계선', fontsize=8, color='gray')

fig.suptitle('Exp D — OOD 탐지 (ID=SAMPLE 10클래스, ODIN vs Mahalanobis)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp_d_ood.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {FIG_DIR}/exp_d_ood.png')
print('† J=2 ODIN sarship=0.000은 단일 trial 이상치 (J=1,3은 정상)')

In [ ]:
# ── Cell 10: figures → Drive 저장 + 결과 커밋 ────────────────────────
%cd /content/repo
import os, shutil
from pathlib import Path

# figures → Drive 복사
FIG_DIR    = Path('/content/repo/results/figures')
DRIVE_FIGS = Path(f'{DRIVE_ROOT}/figures')
DRIVE_FIGS.mkdir(parents=True, exist_ok=True)

for fig_file in FIG_DIR.glob('*.png'):
    dst = DRIVE_FIGS / fig_file.name
    shutil.copy(fig_file, dst)
    print(f'Drive 저장: {dst}')

# Git 커밋
os.system("git config user.email 'colab@sar-atr'")
os.system("git config user.name 'Colab Runner'")
os.system('git add results/figures/ -- ":!*.pth"')
os.system("git commit -m 'exp: A/B/C/D figures 생성' || echo 'nothing to commit'")
os.system('git push')
print('\n완료. Drive figures 폴더에 그래프가 저장됐습니다.')